In [ ]:
import json
import traceback
from typing import List

import pandas as pd
import torch
from tqdm import tqdm

from src.extractors.extractor import (BielikExtractor, DummyExtractor,
                                      OpenAISequentialExtractor,
                                      PllumExtractor)

torch.cuda.empty_cache()

In [ ]:
DEBUG = True
model_name = "speakleash/Bielik-11B-v2.2-Instruct"

In [ ]:
acceptable_models = [
    "speakleash/Bielik-11B-v2.2-Instruct",
    "dummy",
    "CYFRAGOVPL/Llama-PLLuM-8B-instruct",
    "gpt-4o-mini",
]
assert (
    model_name in acceptable_models
), f"Model {model_name} not in acceptable models: {acceptable_models}"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

extractors_map = {
    "speakleash/Bielik-11B-v2.2-Instruct": BielikExtractor,
    "dummy": DummyExtractor,
    "CYFRAGOVPL/Llama-PLLuM-8B-instruct": PllumExtractor,
    "gpt-4o-mini": OpenAISequentialExtractor,
}
extractor = extractors_map[model_name](device=device)
model_size = extractor.get_memory_footprint() #type: ignore
print(f"Model size: {model_size:.2f} GB")

In [ ]:
relations_schema = pd.read_csv("data/edc_baseline/schema.csv", header=None)

relations_description = ""
for i in range(0, len(relations_schema)):
    relation_name, relation_description = relations_schema.iloc[i]
    relations_description += f"{relation_name} - {relation_description}\n"

In [ ]:
with open("data/edc_baseline/oie_few_shot_examples.txt", "r") as file:
    fewshot_examples = "".join(file.readlines())

In [ ]:
with open("data/edc_baseline/dataset.txt", "r") as file:
    dataset = file.readlines()

In [ ]:
def create_system_content(relations_description: str, fewshot_examples: str) -> str:
    return f"""Twoim zadaniem jest wyciągnąć jedną relację łączącą dwa obiekty występujące w tekście, jako trójkę. Trójka musi być w postaci [[Poprzednik, Relacja, Następnik]]. Poprzednik i Następnik są wyrażeniami zapisanymi w tekście. Relacja jest krótkim zapisem związku, jaki łączy Poprzednik i Następnik.
W swojej odpowiedzi przedstaw dokładnie jedną trójkę. Jeśli w tekście jest więcej możliwych trójek, wybierz najardziej prawdopodobną. Nie podawaj żadnych innych informacji czy wyjaśnień.
            
Jedyne relacje, jakie możesz wyciągnąć, to:
{relations_description}

Poniżej przykłady zdań, w których występują obiekty, dla których należy wyciągnąć trójkę:
{fewshot_examples}"""

In [ ]:
def create_prompt_content(sample:str) -> str:
    return f"""Tekst: {sample}
Trójka:"""

In [ ]:

responses: List[str] = []
errors: List[str] = []

system_content = create_system_content(relations_description, fewshot_examples)

subset_size_if_debug = 20

for sample in tqdm(dataset[:subset_size_if_debug] if DEBUG else dataset):
    try:
        print("Sample:", sample.strip())

        template = extractor.create_messages_template( #type: ignore
            system_content, create_prompt_content(sample)
        ) 

        response = extractor.get_response_text(template) #type: ignore
        responses.append(response)
        print("Response:", response)
    except Exception as e:
        tb = traceback.format_exc()
        error_message = f"Error for sample: {sample.strip()}\n{tb}\n"
        errors.append(error_message)
        responses.append("Error\n")
        print("Error:", error_message)

In [ ]:
with open("data/edc_baseline/extracted_relations.json", "w", encoding="utf-8") as file:
    json.dump(dict(responses=responses), file, ensure_ascii=False, indent=2)

with open("data/edc_baseline/errors.json", "w", encoding="utf-8") as file:
    json.dump(dict(errors=errors), file, ensure_ascii=False, indent=2)